# Lumenary — 3D Gaussian Splatting Training

Trains a 3DGS model on a free GPU (~45-60 min including COLMAP).

**SETUP:**
1. Right sidebar → Select **GPU** (L40S on free tier, T4 on Pro)
2. Drag your frames zip into the **Drive** panel (left sidebar)
3. Set the scene name in Step 0, then run all cells in order.

In [ ]:
#@title ## Step 0 — Configuration

SCENE_NAME = "gaborone_city"  #@param ["okavango_delta", "gaborone_city"]

#@markdown Max image dimension (pixels). 1600 is safe for T4/L40S.
MAX_DIM = 1600  #@param [1200, 1600, 1920]

#@markdown Subsample rate. 1=every frame, 2=every other, 3=every 3rd.
SUBSAMPLE_RATE = 1  #@param [1, 2, 3, 4, 5]

#@markdown Training iterations. 30000 is standard.
ITERATIONS = 30000  #@param [15000, 20000, 30000]

print(f"Scene: {SCENE_NAME}")
print(f"Max dimension: {MAX_DIM}px")
print(f"Subsample rate: every {SUBSAMPLE_RATE} frames")
print(f"Training iterations: {ITERATIONS}")

## Step 1 — Install Dependencies

~3-5 minutes.

In [ ]:
#@title ## Step 1 — Install everything (run once)
import subprocess, os, sys, re

# --- Detect CUDA version ---
def get_system_cuda():
    try:
        out = subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT, text=True)
        m = re.search(r'CUDA Version:\s+(\d+)\.(\d+)', out)
        if m: return f"{m.group(1)}.{m.group(2)}"
    except: pass
    try:
        out = subprocess.check_output(['nvcc', '--version'], stderr=subprocess.STDOUT, text=True)
        m = re.search(r'release (\d+)\.(\d+)', out)
        if m: return f"{m.group(1)}.{m.group(2)}"
    except: pass
    return None

system_cuda = get_system_cuda()
print(f"System CUDA: {system_cuda}")

cuda_to_wheel = {"11.8": "cu118", "12.1": "cu121", "12.4": "cu124", "12.6": "cu126", "12.8": "cu128"}
if system_cuda:
    mm = ".".join(system_cuda.split('.')[:2])
    torch_cuda = cuda_to_wheel.get(mm, "cu128" if system_cuda.startswith("13.") else "cu126")
else:
    torch_cuda = "cu126"
print(f"PyTorch wheel: {torch_cuda}")

!sudo apt-get update -qq && sudo apt-get install -y -qq ninja-build colmap libglm-dev > /dev/null 2>&1
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/{torch_cuda}
!pip install -q plyfile Pillow tqdm opencv-python joblib

GS_DIR = os.path.expanduser("~/gaussian-splatting")
if not os.path.exists(GS_DIR):
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting {GS_DIR}

os.chdir(GS_DIR)
import torch as _torch
_cc = _torch.cuda.get_device_capability(0) if _torch.cuda.is_available() else (7, 5)
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{_cc[0]}.{_cc[1]}"
print(f"GPU compute capability: {_cc[0]}.{_cc[1]}")
!pip install --no-build-isolation -q submodules/diff-gaussian-rasterization
!pip install --no-build-isolation -q submodules/simple-knn
!pip install --no-build-isolation -q submodules/fused-ssim

os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'

import torch
print(f"\n{'='*50}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
try:
    import diff_gaussian_rasterization
    print("diff-gaussian-rasterization: OK")
except ImportError as e:
    print(f"ERROR: {e}")
print(f"{'='*50}")

## Step 2 — Upload & Prepare Images

Drag your zip into the **Drive** panel (left sidebar).

Then run this cell.

In [ ]:
#@title ## Step 2 — Find and process uploaded images
import os, zipfile, shutil, hashlib, sys, glob
from PIL import Image

if 'SCENE_NAME' not in dir(): SCENE_NAME = "gaborone_city"
if 'SUBSAMPLE_RATE' not in dir(): SUBSAMPLE_RATE = 1
if 'MAX_DIM' not in dir(): MAX_DIM = 1600
if 'ITERATIONS' not in dir(): ITERATIONS = 30000

HOME = os.path.expanduser("~")
INPUT_DIR = os.path.join(HOME, "input_raw")
PREPARED_DIR = os.path.join(HOME, "input_images")

# Clear stale data from previous runs
for _d in [INPUT_DIR, PREPARED_DIR]:
    if os.path.exists(_d): shutil.rmtree(_d)
    os.makedirs(_d, exist_ok=True)

image_exts = ('.jpg', '.jpeg', '.png', '.webp')

# Search for zip in common locations
zip_candidates = (
    glob.glob(os.path.join(HOME, f"*{SCENE_NAME}*.zip")) +
    glob.glob(os.path.join(HOME, "*.zip")) +
    glob.glob("/teamspace/datasets/*/*.zip") +
    glob.glob("/teamspace/datasets/*.zip") +
    glob.glob("/tmp/*.zip")
)

if not zip_candidates:
    print(f"ERROR: No zip file found.")
    print(f"\nDrag your zip (e.g. {SCENE_NAME}_3fps.zip) into the Drive panel.")
    sys.exit(1)

zip_path = zip_candidates[0]
print(f"Found: {os.path.basename(zip_path)}")

# Extract
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(INPUT_DIR)

# Find and flatten images
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f.lower().endswith(image_exts):
            src = os.path.join(root, f)
            dst = os.path.join(PREPARED_DIR, f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)

frames = sorted([f for f in os.listdir(PREPARED_DIR) if f.lower().endswith(image_exts)])
print(f"Total images: {len(frames)}")

# Remove duplicates
seen = set()
unique = []
for f in frames:
    h = hashlib.md5(open(os.path.join(PREPARED_DIR, f), 'rb').read()).hexdigest()
    if h not in seen:
        seen.add(h)
        unique.append(f)
removed = len(frames) - len(unique)
if removed > 0: print(f"Removed {removed} duplicates")
frames = unique

# Remove blurry frames
sizes = [(f, os.path.getsize(os.path.join(PREPARED_DIR, f))) for f in frames]
avg = sum(s for _, s in sizes) / len(sizes)
quality = [f for f, s in sizes if s > avg * 0.4]
removed = len(frames) - len(quality)
if removed > 0: print(f"Removed {removed} blurry frames")
frames = quality

if len(frames) == 0:
    print("ERROR: No images remaining.")
    sys.exit(1)

# Subsample
if SUBSAMPLE_RATE > 1:
    frames = frames[::SUBSAMPLE_RATE]
    print(f"Subsampled to {len(frames)} frames")

# Resize
resized = 0
for f in frames:
    src = os.path.join(PREPARED_DIR, f)
    img = Image.open(src)
    mx = max(img.size)
    if mx > MAX_DIM:
        r = MAX_DIM / mx
        img.resize((int(img.size[0]*r), int(img.size[1]*r)), Image.LANCZOS).save(src, quality=95)
        resized += 1
if resized: print(f"Resized {resized} images")

print(f"\n{'='*50}")
print(f"READY: {len(frames)} images for training")
sample = Image.open(os.path.join(PREPARED_DIR, frames[0]))
print(f"Sample: {frames[0]} ({sample.size[0]}x{sample.size[1]})")
print(f"{'='*50}")

## Step 3 — Preview Images

In [ ]:
#@title ## Step 3 — Preview
from IPython.display import display, HTML
from PIL import Image
import os, base64

HOME = os.path.expanduser("~")
PREPARED_DIR = os.path.join(HOME, "input_images")
image_exts = ('.jpg', '.jpeg', '.png', '.webp')
frames = sorted([f for f in os.listdir(PREPARED_DIR) if f.lower().endswith(image_exts)])

html = "<div style='display:flex;flex-wrap:wrap;gap:4px'>"
for f in frames[:30]:
    img = Image.open(os.path.join(PREPARED_DIR, f))
    img.thumbnail((150, 150))
    tmp = f"/tmp/thumb_{f}"
    img.save(tmp)
    with open(tmp, 'rb') as fh:
        b64 = base64.b64encode(fh.read()).decode()
    html += f"<img src='data:image/jpeg;base64,{b64}' style='height:120px;margin:2px'>"
html += "</div>"
display(HTML(html))
print(f"{min(30, len(frames))} of {len(frames)} images shown")

## Step 4 — Run COLMAP (~5-15 min)

In [ ]:
#@title ## Step 4 — COLMAP
import subprocess, os, shutil, struct

HOME = os.path.expanduser("~")
PREPARED_DIR = os.path.join(HOME, "input_images")
image_exts = ('.jpg', '.jpeg', '.png', '.webp')
WS = os.path.join(HOME, "colmap_ws")
os.makedirs(os.path.join(WS, "sparse"), exist_ok=True)

db = os.path.join(WS, "database.db")
imgs = os.path.join(WS, "images")
if os.path.islink(imgs):
    os.unlink(imgs)
elif os.path.isdir(imgs):
    shutil.rmtree(imgs)
os.symlink(PREPARED_DIR, imgs)
if os.path.exists(db): os.remove(db)

n = len([f for f in os.listdir(imgs) if f.lower().endswith(image_exts)])
print(f"COLMAP on {n} images...")

os.environ['QT_QPA_PLATFORM'] = 'offscreen'

print("[1/3] Features...")
!QT_QPA_PLATFORM=offscreen colmap feature_extractor --database_path {db} --image_path {imgs} --ImageReader.camera_model OPENCV --SiftExtraction.use_gpu 0
print("[2/3] Matching...")
!QT_QPA_PLATFORM=offscreen colmap exhaustive_matcher --database_path {db} --SiftMatching.use_gpu 0
print("[3/3] Mapping...")
!QT_QPA_PLATFORM=offscreen colmap mapper --database_path {db} --image_path {imgs} --output_path {WS}/sparse

s0 = os.path.join(WS, "sparse", "0")
if os.path.exists(s0):
    ib = os.path.join(s0, 'images.bin')
    if os.path.exists(ib):
        with open(ib, 'rb') as f: reg = struct.unpack('Q', f.read(8))[0]
        print(f"\nCOLMAP OK: {reg}/{n} cameras registered")
        if reg < n * 0.7: print(f"WARNING: Low registration. Images may be too similar.")
else:
    print("ERROR: COLMAP failed.")

## Step 5 — Train (~30-45 min)

In [ ]:
#@title ## Step 5 — Train
import torch, os

ITERATIONS = 30000 if 'ITERATIONS' not in dir() else ITERATIONS
HOME = os.path.expanduser("~")
WS = os.path.join(HOME, "colmap_ws")

if torch.cuda.is_available():
    free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f"VRAM: {free / 1e9:.1f} GB free")

os.chdir(os.path.join(HOME, "gaussian-splatting"))
!python train.py -s {WS} -m {HOME}/output_model --iterations {ITERATIONS}
print("Training complete!")

## Step 6 — Export PLY

In [ ]:
#@title ## Step 6 — Export
import os, shutil

ITERATIONS = 30000 if 'ITERATIONS' not in dir() else ITERATIONS
SCENE_NAME = 'gaborone_city' if 'SCENE_NAME' not in dir() else SCENE_NAME
HOME = os.path.expanduser("~")
PLY = os.path.join(HOME, "output_model", "point_cloud", f"iteration_{ITERATIONS}", "point_cloud.ply")

if os.path.exists(PLY):
    mb = os.path.getsize(PLY) / 1e6
    print(f"PLY ready: {mb:.1f} MB")
    out = os.path.join(HOME, f"{SCENE_NAME}_point_cloud.ply")
    shutil.copy2(PLY, out)
    print(f"Saved to: {out}")
    print(f"\nDownload this file from the Drive panel.")
    print(f"Then upload to GCS:")
    print(f"  gsutil cp {SCENE_NAME}_point_cloud.ply gs://lumenary-viewer-us/scenes/{SCENE_NAME}/scene.ply")
else:
    print(f"ERROR: PLY not found at {PLY}")